In [1]:
pip install -U scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/ef/0e/97dbca66347b8cf0ea8b529e6bb9367e337ba2e8be0ef5c1a545232abfde/scikit_learn-1.7.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.1.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 60.9 MB/s eta 0:00:00 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [59]:
!pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Obtaining dependency information for lightgbm from https://files.pythonhosted.org/packages/42/86/dabda8fbcb1b00bcfb0003c3776e8ade1aa7b413dff0a2c08f457dace22f/lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 27.7 MB/s eta 0:00:000:00:010:00:01


In [74]:
!pip install catboost

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Obtaining dependency information for catboost from https://files.pythonhosted.org/packages/e2/47/abee19aae4b2a2a21e40e3c09db784099d189b3a0745e59c1d152700d90a/catboost-1.2.8-cp311-cp311-manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for graphviz from https://files.pythonhosted.org/packages/91/4c/e0ce1ef95d4000ebc1c11801f9b944fa5910ecc15b5e351865763d8657f8/graphviz-0.21-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 12.1 MB/s eta 0:00:00 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 846.7 kB/s eta 0:00:009 MB/s eta 0:00:01


In [4]:
import pandas as pd
import numpy as np
import re

In [34]:
df = pd.read_csv("appartments_train.csv")
test_df = pd.read_csv("appartments_test.csv")
df.head()

,id,address,layout,price,gps_lat,gps_lon,area,floor,total_floors,construction,...,poi_school_kindergarten_nearest,poi_transport_count,poi_transport_nearest,poi_grocery_count,poi_grocery_nearest,poi_restaurant_count,poi_restaurant_nearest,text,first_seen,last_seen
0,6980,"Tavolníková, Praha - Krč",2+kk,6450000.0,50.019823,14.461106,57.0,5,NaN,Panelová,...,290.0,5,189.0,5,160.0,5,344.0,Nabízím k prodeji byt 2+kk/45 m2 s chodbou pře...,2025-08-20,2025-08-23
1,1860,"Pitterova, Praha 3 - Žižkov",3+kk,14750000.0,50.085597,14.468936,93.0,2,12.0,Smíšená,...,211.0,5,70.0,5,324.0,5,225.0,Nabízíme k prodeji světlý a prostorný byt 3+kk...,2025-06-27,2025-07-08
2,5225,"Perucká, Praha 2 - Vinohrady",1+kk,5490000.0,50.068807,14.437938,27.0,2,4.0,Cihlová,...,179.0,5,176.0,5,377.0,5,152.0,"Prodej bytu 1+kk o velikosti 27m², umístěného ...",2025-07-16,2025-08-12
3,4441,"Brožíkova, Praha - Košíře",1+1,6352000.0,50.071383,14.388532,38.0,2,NaN,Cihlová,...,220.0,5,84.0,5,46.0,5,110.0,Přímý vlastník nabízí k prodeji zrekonstruovan...,2025-07-01,2025-07-31
4,7143,"Hnězdenská, Praha 8 - Troja",2+kk,6190000.0,50.126809,14.423477,46.0,11,12.0,Smíšená,...,258.0,5,178.0,5,121.0,5,153.0,Připravujeme k prodeji byt 2 + kk po rekonstru...,2025-08-22,2025-09-01


In [90]:
test_df = pd.read_csv("appartments_test.csv")

Preprocessing. Instead of 2 columns first_seen and last_seen we want to make 1 column listing_duration_days that contains the difference between them. 

In [35]:
if "first_seen" in df.columns and "last_seen" in df.columns:
    df["first_seen"] = pd.to_datetime(df["first_seen"], errors="coerce")
    df["last_seen"] = pd.to_datetime(df["last_seen"], errors="coerce")
    df["listing_duration_days"] = (df["last_seen"] - df["first_seen"]).dt.days

Creating a function for parsing the layout. We will extract from it the number of rooms, number of kitchens, and the kitchen type

In [36]:
def parse_layout_series(srs):
    rooms = []
    kitchens = []
    kitchen_type = []   # 0 = none/unknown, 1 = kitchen, 2 = kitchenette (kk)
    for v in srs.fillna("").astype(str):
        raw = v.strip().lower().replace(" ", "")
        if raw == "" or raw == "nan":
            rooms.append(np.nan)
            kitchens.append(np.nan)
            kitchen_type.append(np.nan)
            continue
        # Split only once so we keep "extra info" in suffix, e.g. "3+kkb"
        parts = raw.split("+", 1)
        # Parse rooms (before '+')
        try:
            r = int(parts[0])
        except:
            r = np.nan
        rooms.append(r)
        # If there's no suffix, mark as no kitchen info
        if len(parts) == 1 or parts[1] == "":
            kitchens.append(0)
            kitchen_type.append(0)
            continue
        suffix = parts[1]
        # If suffix explicitly contains "kk" => kitchenette
        if "kk" in suffix:
            # if there is a leading number like "2kk" treat it as that number of kk (rare)
            m_leading_num = re.match(r"^(\d+)", suffix)
            if m_leading_num:
                kitchens.append(int(m_leading_num.group(1)))
            else:
                kitchens.append(1)
            kitchen_type.append(2)
            continue
        # If suffix is purely digits (e.g., "1", "2") => number of kitchens
        if re.fullmatch(r"\d+", suffix):
            kitchens.append(int(suffix))
            kitchen_type.append(1)
            continue
        # If suffix contains pattern like "2k" (digits + k)
        m = re.search(r"(\d+)k", suffix)
        if m:
            kitchens.append(int(m.group(1)))
            kitchen_type.append(1)
            continue
        # If suffix contains 'k' anywhere (e.g., 'k', 'k+bal', 'kbal') -> one kitchen
        if "k" in suffix:
            kitchens.append(1)
            kitchen_type.append(1)
            continue
        # If suffix contains a digit somewhere (e.g., '1m', '1b') we still interpret as kitchens
        m_any_digit = re.search(r"(\d+)", suffix)
        if m_any_digit:
            kitchens.append(int(m_any_digit.group(1)))
            kitchen_type.append(1)
            continue
        # Otherwise: no kitchen info found
        kitchens.append(0)
        kitchen_type.append(0)

    return pd.DataFrame({
        "layout_rooms": rooms,
        "layout_kitchens": kitchens,
        "layout_kitchen_type": kitchen_type
    })

In [37]:
parse_layout_series(df["layout"]).head()

,layout_rooms,layout_kitchens,layout_kitchen_type
0,2,1,2
1,3,1,2
2,1,1,2
3,1,1,1
4,2,1,2


In [38]:
df["layout"].head()

0    2+kk
1    3+kk
2    1+kk
3     1+1
4    2+kk
Name: layout, dtype: object

In [39]:
parsed_layout = parse_layout_series(df["layout"])

# Add the new columns
df = pd.concat([df, parsed_layout], axis=1)

# Drop old layout string column
df = df.drop(columns=["layout"])

df.head()

,id,address,price,gps_lat,gps_lon,area,floor,total_floors,construction,condition,...,poi_grocery_nearest,poi_restaurant_count,poi_restaurant_nearest,text,first_seen,last_seen,listing_duration_days,layout_rooms,layout_kitchens,layout_kitchen_type
0,6980,"Tavolníková, Praha - Krč",6450000.0,50.019823,14.461106,57.0,5,NaN,Panelová,Velmi dobrý,...,160.0,5,344.0,Nabízím k prodeji byt 2+kk/45 m2 s chodbou pře...,2025-08-20,2025-08-23,3,2,1,2
1,1860,"Pitterova, Praha 3 - Žižkov",14750000.0,50.085597,14.468936,93.0,2,12.0,Smíšená,Průměrný,...,324.0,5,225.0,Nabízíme k prodeji světlý a prostorný byt 3+kk...,2025-06-27,2025-07-08,11,3,1,2
2,5225,"Perucká, Praha 2 - Vinohrady",5490000.0,50.068807,14.437938,27.0,2,4.0,Cihlová,Průměrný,...,377.0,5,152.0,"Prodej bytu 1+kk o velikosti 27m², umístěného ...",2025-07-16,2025-08-12,27,1,1,2
3,4441,"Brožíkova, Praha - Košíře",6352000.0,50.071383,14.388532,38.0,2,NaN,Cihlová,Průměrný,...,46.0,5,110.0,Přímý vlastník nabízí k prodeji zrekonstruovan...,2025-07-01,2025-07-31,30,1,1,1
4,7143,"Hnězdenská, Praha 8 - Troja",6190000.0,50.126809,14.423477,46.0,11,12.0,Smíšená,Velmi dobrý,...,121.0,5,153.0,Připravujeme k prodeji byt 2 + kk po rekonstru...,2025-08-22,2025-09-01,10,2,1,2


In [40]:
df["layout_kitchen_type"] = df["layout_kitchen_type"].astype("category")

In [41]:
numeric_cols = [
    "area", "balcony_area", "garden_area",
    "floor", "total_floors", "parking",
    "poi_transport_nearest", "poi_grocery_nearest",
    "listing_duration_days",
    "layout_rooms",        
    "layout_kitchens"      
]
categorical_cols = [
    "construction", "condition",
    "ownership", "elevator",
    "layout_kitchen_type"   
]
coords = ["gps_lat", "gps_lon"]

In [42]:
keep_cols = (
    numeric_cols
    + categorical_cols
    + coords
    + ["price"]       
)

In [43]:
df = df[keep_cols].copy()

In [44]:
df.head()

,area,balcony_area,garden_area,floor,total_floors,parking,poi_transport_nearest,poi_grocery_nearest,listing_duration_days,layout_rooms,layout_kitchens,construction,condition,ownership,elevator,layout_kitchen_type,gps_lat,gps_lon,price
0,57.0,NaN,NaN,5,NaN,NaN,189.0,160.0,3,2,1,Panelová,Velmi dobrý,Družstevní,Yes,2,50.019823,14.461106,6450000.0
1,93.0,57.0,NaN,2,12.0,1.0,70.0,324.0,11,3,1,Smíšená,Průměrný,Osobní,Yes,2,50.085597,14.468936,14750000.0
2,27.0,NaN,NaN,2,4.0,NaN,176.0,377.0,27,1,1,Cihlová,Průměrný,Osobní,NaN,2,50.068807,14.437938,5490000.0
3,38.0,NaN,NaN,2,NaN,NaN,84.0,46.0,30,1,1,Cihlová,Průměrný,Osobní,NaN,1,50.071383,14.388532,6352000.0
4,46.0,NaN,NaN,11,12.0,1.0,178.0,121.0,10,2,1,Smíšená,Velmi dobrý,Osobní,Yes,2,50.126809,14.423477,6190000.0


In [45]:
# Check for missing values
print("Missing values per column:\n")
print(df.isnull().sum())

print("\nAny missing values in the dataset?", df.isnull().values.any())

Missing values per column:

area                        0
balcony_area             2580
garden_area              5000
floor                       0
total_floors             1642
parking                  3972
poi_transport_nearest     174
poi_grocery_nearest       176
listing_duration_days       0
layout_rooms                0
layout_kitchens             0
construction                0
condition                   0
ownership                   0
elevator                 1525
layout_kitchen_type         0
gps_lat                     0
gps_lon                     0
price                       0
dtype: int64

Any missing values in the dataset? True


Handling missing values: 3 strategies : 
A) filling with zeroes
B) filling with mean 
C) filling with missing for categorical data

In [46]:
#A) Zero-fill numerical features that represent "not present"
zero_impute_cols = ["cellar_area", "balcony_area", "garden_area", "parking"]
for col in zero_impute_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# B) Standard numeric features → median
median_impute_cols = [
    "total_floors",
    "poi_doctors_nearest", "poi_leisure_time_nearest",
    "poi_school_kindergarten_nearest", "poi_transport_nearest",
    "poi_grocery_nearest", "poi_restaurant_nearest",
    "listing_duration_days"
]

for col in median_impute_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# C) Categorical → "missing"
categorical_cols = [ "elevator"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("missing").astype("category")

Preprocessing for linear regression

In [52]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# NUMERIC pipeline
numeric_transformer_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# CATEGORICAL pipeline
categorical_transformer_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# FULL PREPROCESSOR
linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_linear, coords + numeric_cols),
        ("cat", categorical_transformer_linear, categorical_cols)
    ],
    remainder="drop"
)

In [53]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("preprocess", linear_preprocessor),
    ("model", LinearRegression())
])

In [49]:
from sklearn.model_selection import train_test_split


y = df["price"]
X = df.drop(columns=["price"])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=42,     
    shuffle=True
)    

In [54]:
linear_model.fit(X_train , y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [56]:
from sklearn.metrics import mean_absolute_percentage_error
preds = linear_model.predict(X_test)
mape = mean_absolute_percentage_error(y_test, preds)
print("MAPE:", mape)
print("MAPE %:", mape * 100)

MAPE: 0.19082467068530454
MAPE %: 19.082467068530455


Preprocessing for LightGBM

In [69]:
def preprocess_for_lgbm(df):
    df = df.copy()
    
    # 1) Structural zero-fill
    zero_impute_cols = ["cellar_area", "balcony_area", "garden_area", "parking"]
    for col in zero_impute_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 2) Median-impute numeric features
    median_impute_cols = [
        "area", "total_floors", "floor",
        "poi_doctors_nearest", "poi_leisure_time_nearest",
        "poi_school_kindergarten_nearest", "poi_transport_nearest",
        "poi_grocery_nearest", "poi_restaurant_nearest",
        "listing_duration_days",
        "gps_lat", "gps_lon",
        "layout_rooms", "layout_kitchens"
    ]
    for col in median_impute_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # 3) Safe categorical handling
    categorical_cols = [
        "construction", "condition", "ownership",
        "elevator", "layout_kitchen_type"
    ]
    
    for col in categorical_cols:
        if col in df.columns:
            # Convert to plain strings first
            df[col] = df[col].astype("string").fillna("missing")
            # Then convert to pandas category dtype
            df[col] = df[col].astype("category")

    return df

In [83]:
import lightgbm as lgb

df_lgbm = preprocess_for_lgbm(df)

X = df_lgbm.drop(columns=["price"])
y = df_lgbm["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols)


In [84]:
from lightgbm import LGBMRegressor
from lightgbm.callback import early_stopping

model = LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_data_in_leaf=30,
    n_estimators=500,             # like num_boost_round
)

# Use callbacks parameter instead of early_stopping_rounds
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="mape",
    callbacks=[early_stopping(stopping_rounds=30)]  # Use early_stopping callback
)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Auto-choosing row-wise multi-threading, 

,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [85]:
preds=model.predict(X_test)
mae=mean_absolute_error(y_test,preds)
rmse=np.sqrt(mean_squared_error(y_test,preds))
mape=mean_absolute_percentage_error(y_test,preds)
r2=r2_score(y_test,preds)
print(mae,rmse,mape,r2)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
1041887.7149506884 1540263.968308672 0.10993774238063861 0.8222703575599252


Way better then what we achieved with linear regression but lets try catboost

In [75]:
y = df["price"]
X = df.drop(columns=["price"])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [80]:
categorical_cols = [
        "construction", "condition", "ownership",
        "elevator", "layout_kitchen_type"
    ]

In [81]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    depth=8,
    learning_rate=0.05,
    iterations=2000,
    loss_function="RMSE",
    eval_metric="MAPE",
    random_seed=42,
    verbose=200,
)

cat_feature_indices = [X_train.columns.get_loc(col) for col in categorical_cols]

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    cat_features=cat_feature_indices,
    verbose=200
)

0:	learn: 0.3270930	test: 0.3254210	best: 0.3254210 (0)	total: 66.2ms	remaining: 2m 12s
200:	learn: 0.0927917	test: 0.1125693	best: 0.1125693 (200)	total: 2.17s	remaining: 19.4s
400:	learn: 0.0767262	test: 0.1093858	best: 0.1093771 (397)	total: 4.3s	remaining: 17.1s
600:	learn: 0.0655068	test: 0.1073007	best: 0.1073007 (600)	total: 6.39s	remaining: 14.9s
800:	learn: 0.0563398	test: 0.1068444	best: 0.1068324 (798)	total: 8.59s	remaining: 12.9s
1000:	learn: 0.0495817	test: 0.1064568	best: 0.1064053 (997)	total: 10.7s	remaining: 10.7s
1200:	learn: 0.0437433	test: 0.1059836	best: 0.1059691 (1193)	total: 17.9s	remaining: 11.9s
1400:	learn: 0.0385086	test: 0.1059375	best: 0.1058420 (1296)	total: 21.2s	remaining: 9.04s
1600:	learn: 0.0342775	test: 0.1059824	best: 0.1058420 (1296)	total: 24s	remaining: 5.99s
1800:	learn: 0.0306995	test: 0.1059821	best: 0.1058420 (1296)	total: 26.6s	remaining: 2.93s
1999:	learn: 0.0275718	test: 0.1060643	best: 0.1058420 (1296)	total: 28.8s	remaining: 0us

bestT

In [82]:
preds = model.predict(X_test)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"MAPE: {mape:.4f} ({mape * 100:.2f}%)")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAPE: 0.1058 (10.58%)
MAE: 1003266.5838730378
RMSE: 1485516.714105315
R²: 0.8346802878930718


We see a slight improvement in metrics so we'll use catboost for our prediction. But first of all we need to preprocess our test dataset the same way as we did with the train one  

In [130]:
test_df = pd.read_csv("appartments_test.csv")
print(test_df.shape)

(1020, 32)


In [131]:
parsed_layout = parse_layout_series(test_df["layout"])

# Add the new columns
test_df = pd.concat([test_df, parsed_layout], axis=1)

# Drop old layout string column
test_df = test_df.drop(columns=["layout"])

test_df.head()

,id,address,price,gps_lat,gps_lon,area,floor,total_floors,construction,condition,...,poi_grocery_count,poi_grocery_nearest,poi_restaurant_count,poi_restaurant_nearest,text,first_seen,last_seen,layout_rooms,layout_kitchens,layout_kitchen_type
0,8795,"Sokolovská, Praha 9 - Libeň",NaN,50.106360,14.487984,57.0,2,NaN,Cihlová,Špatný,...,5,67.0,5,48.0,Nabízíme k prodeji byt 2+kk v atraktivní lokal...,2025-09-17,2025-10-17,2,1,2
1,6516,"Dubrovnická, Praha 5 - Košíře",NaN,50.066717,14.368126,65.0,3,3.0,Panelová,Špatný,...,5,382.0,5,370.0,"Prodej bytu 3+1, 65 m2 + lodžie 7 m2 – ul. Dub...",2025-08-11,2025-08-11,3,1,2
2,4714,"Kbelská, Praha 9 - Hloubětín",NaN,50.105305,14.530936,31.0,4,4.0,Cihlová,Průměrný,...,5,30.0,5,56.0,"Cena bytu platí pro zájemce s hotovostí, pro o...",2025-07-07,2025-11-04,1,1,2
3,8423,"Habrová, Praha 3 - Žižkov",NaN,50.088656,14.494247,53.0,7,9.0,Panelová,Špatný,...,5,459.0,5,293.0,Byt 2+1 s lodžií a sklepem na Žižkově – přílež...,2025-09-12,2025-11-04,2,1,1
4,5361,"Hudečkova, Praha 4 - Podolí",NaN,50.042159,14.433814,50.0,2,11.0,Smíšená,Průměrný,...,5,964.0,5,608.0,Bydlete v Podolí! Ve výhradním zastoupení maji...,2025-07-18,2025-11-04,2,1,2


In [132]:
if "first_seen" in test_df.columns and "last_seen" in test_df.columns:
    test_df["first_seen"] = pd.to_datetime(test_df["first_seen"], errors="coerce")
    test_df["last_seen"] = pd.to_datetime(test_df["last_seen"], errors="coerce")
    test_df["listing_duration_days"] = (test_df["last_seen"] - test_df["first_seen"]).dt.days

In [133]:
test_df = test_df[keep_cols].copy()

In [134]:
# Check for missing values
print("Missing values per column:\n")
print(test_df.isnull().sum())

print("\nAny missing values in the dataset?", test_df.isnull().values.any())

Missing values per column:

area                        0
balcony_area              536
garden_area              1020
floor                       0
total_floors              338
parking                   805
poi_transport_nearest      35
poi_grocery_nearest        35
listing_duration_days       0
layout_rooms                0
layout_kitchens             0
construction                0
condition                   0
ownership                   0
elevator                  298
layout_kitchen_type         0
gps_lat                     0
gps_lon                     0
price                    1020
dtype: int64

Any missing values in the dataset? True


In [135]:
test_df = preprocess_for_lgbm(test_df)

In [136]:
print(test_df.isnull().sum())

area                        0
balcony_area                0
garden_area                 0
floor                       0
total_floors                0
parking                     0
poi_transport_nearest       0
poi_grocery_nearest         0
listing_duration_days       0
layout_rooms                0
layout_kitchens             0
construction                0
condition                   0
ownership                   0
elevator                    0
layout_kitchen_type         0
gps_lat                     0
gps_lon                     0
price                    1020
dtype: int64


In [127]:
categorical_cols = [
        "construction", "condition", "ownership",
        "elevator", "layout_kitchen_type"
    ]

In [137]:
X = test_df.drop(columns=["price"])

In [139]:
test_preds = model.predict(X)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5


In [140]:
orig_test = pd.read_csv("appartments_test.csv")
submission = pd.DataFrame({
    "id": orig_test["id"],   # if present
    "predicted_price": test_preds
})

submission.to_csv("predictions.csv", index=False)
print("Saved predictions.csv")

Saved predictions.csv
